In [1]:
# Packages, seed and path
## packages
from edge_sim_py import *
import math
import os
import random
import msgpack
import pandas as pd
import json
import numpy as np
from numpy import random

## path
import os
abspath = os.path.abspath(os.path.join('..'))
algo_name = "faticanti"

In [2]:
# change the slas
def change_slas(path, mu, sigma):
    # Opening JSON file
    f = open(path)
    
    # returns JSON object as 
    # a dictionary
    data = json.load(f)
    
    # Iterating through the json
    # sla values from probabilistic distribution
    n = len(data['User'])
    sla_values = np.around(np.random.normal(mu, sigma, n), decimals=0)
    # list
    for i in range(len(data['User'])):
        print('>>> user', i, "<<<")
        print('before:', data['User'][i].get('attributes').get('delay_slas'))
        data['User'][i].get('attributes').get('delay_slas').update({str(i+1): sla_values[i]})
        print('after:', data['User'][i].get('attributes').get('delay_slas'))

    # Closing file
    f.close()

    # Serializing json
    json_object = json.dumps(data, indent=4)
    
    # Writing to sample.json
    with open(path, "w") as outfile:
        outfile.write(json_object)

# change the network delays
def change_network(path, link_delay, wireless_delay):
    # Opening JSON file
    f = open(path)
    
    # returns JSON object as 
    # a dictionary
    data = json.load(f)
    
    # Iterating through the json - networklink
    for i in range(len(data['NetworkLink'])):
        print('>>> network link', i, "<<<")
        print('before:', data['NetworkLink'][i].get('attributes').get('delay'))
        data['NetworkLink'][i].get('attributes').update({'delay': link_delay})
        print('after:', data['NetworkLink'][i].get('attributes').get('delay'))

    # Iterating through the json - wireless
    for i in range(len(data['BaseStation'])):
        print('>>> base station', i, "<<<")
        print('before:', data['BaseStation'][i].get('attributes').get('wireless_delay'))
        data['BaseStation'][i].get('attributes').update({'wireless_delay': wireless_delay})
        print('after:', data['BaseStation'][i].get('attributes').get('wireless_delay'))
    # Closing file
    f.close()

    # Serializing json
    json_object = json.dumps(data, indent=4)
    
    # Writing to sample.json
    with open(path, "w") as outfile:
        outfile.write(json_object)

In [3]:
# Custom collect method to measure users
def user_custom_collect_method(self) -> dict: 
    # Python libraries
    import copy
    """Method that collects a set of metrics for the object.

    Returns:
        metrics (dict): Object metrics.
    """
    access_history = {}
    for app in self.applications:
        access_history[str(app.id)] = self.access_patterns[str(app.id)].history

    try: 
        delay_sla_deadlines = self.delay_slas.get(str(self.id)) - self.delays.get(str(self.id)) 
    except: 
         delay_sla_deadlines = None

    metrics = {
        "Instance ID": self.id,
        "Coordinates": self.coordinates,
        "Base Station": f"{self.base_station} ({self.base_station.coordinates})" if self.base_station else None,
        "Delays": self.delays.get(str(self.id)), #copy.deepcopy(self.delays),
        "Delay slas": self.delay_slas.get(str(self.id)),#copy.deepcopy(self.delay_slas),
        "Delay sla deadlines": delay_sla_deadlines,
        "Communication Paths": copy.deepcopy(self.communication_paths),
        "Making Requests": copy.deepcopy(self.making_requests),
        "Access History": copy.deepcopy(access_history)
    }
    return metrics


def server_custom_collect_method(self) -> dict:
        """Method that collects a set of metrics for the object.

        Returns:
            metrics (dict): Object metrics.
        """
        metrics = {
            "Instance ID": self.id,
            "Coordinates": self.coordinates,
            "Available": self.available,
            "CPU": self.cpu,
            "RAM": self.memory,
            "Disk": self.disk,
            "CPU Demand": self.cpu_demand,
            "RAM Demand": self.memory_demand,
            "Disk Demand": self.disk_demand,
            "Ongoing Migrations": self.ongoing_migrations,
            "Services": [service.id for service in self.services],
            "Registries": [registry.id for registry in self.container_registries],
            "Layers": [layer.instruction for layer in self.container_layers],
            "Images": [image.name for image in self.container_images],
            "Download Queue": [f.metadata["object"].instruction for f in self.download_queue],
            "Waiting Queue": [layer.instruction for layer in self.waiting_queue],
            "Max. Concurrent Layer Downloads": self.max_concurrent_layer_downloads,
            "Power Consumption": self.get_power_consumption(),
            "rfc": self.rfc,
            "rf_cpu": self.rf_cpu,
            "rf_memory": self.rf_memory,
        }
        return metrics

In [4]:
def faticanti(parameters: dict = {}):
    """Adapted version of [1], that focuses on finding host servers for microservice-based applications on
    Edge Computing scenarios with multiple infrastructure providers. This heuristic was originally designed to define
    service's deployment (i.e., initial placement). The code below is an adapted version that uses the heuristic's
    original reasoning to perform service migration when it detects application's SLA are violated.

    References:
        [1] Faticanti, Francescomaria, et al. "Deployment of Application Microservices in Multi-Domain Federated
        Fog Environments." International Conference on Omni-layer Intelligent Systems (COINS). IEEE, 2020.
    """

    # Based on Faticanti's idea, we sort services based on their positions in their application's service chain and
    # according to their applications's network demand (services from applications with lower demand come first).
    # apps = sorted(Application.all(), key=lambda app: (app.services[0].cpu_demand), reverse=False)

    print(f"[STEP {parameters['current_step']}]")

    for edge_server in EdgeServer.all():
        # reset fragmentation monitoring
        edge_server.rfc = 0
        edge_server.rf_cpu = 0
        edge_server.rf_memory = 0
    
    apps = sorted(
        Application.all(), 
        key=lambda app: (app.services[0].cpu_demand),
        reverse=False
        )

    for app in apps:
        user = app.users[0]
        #services = sorted(app.services, key=lambda s: (-s.privacy_requirement))
        services = app.services

        # FUNCTION: get_host_candidates(user=user)
        user_switch = user.base_station.network_switch
        host_candidates = []
        for edge_server in EdgeServer.all():

            topology = user_switch.model.topology
            path = find_shortest_path(origin_network_switch=user_switch, target_network_switch=edge_server.network_switch)
            delay = topology.calculate_path_delay(path=path)
            
            host_candidates.append(
                {
                    "object": edge_server,
                    "delay": delay,
                    "trust_degree": user.providers_trust[str(edge_server.infrastructure_provider)],
                }
            )

        edge_servers = sorted(host_candidates, key=lambda s: (s["delay"]))

        for service in services:
            # Greedily iterating over the list of edge servers to find a host for the service
            for edge_server_metadata in edge_servers:
                edge_server = edge_server_metadata["object"]
                # Check if the edge server has resources to host the service
                if edge_server.has_capacity_to_host(service=service):
                    # We just need to migrate the service if it's not already in the least occupied edge server
                    service.drop = False
                    if service.server != edge_server:
                        
                        #print(f"Migrating {service} From {service.server} to {edge_server}")
                        service.provision(target_server=edge_server)
                        # After start migrating the service we can move on to the next service
                        break
                else:
                    if ((edge_server.cpu - service.cpu_demand) >= 0) | ((edge_server.memory - service.memory_demand) >= 0):
                        
                        edge_server.rfc = edge_server.rfc + 1

                        #print('entrou', edge_server, edge_server.rfc)
                        
                        if ((edge_server.cpu - service.cpu_demand) >= 0):
                            edge_server.rf_cpu = edge_server.rf_cpu + (edge_server.cpu - service.cpu_demand)
                            #print(edge_server.rf_cpu)
                        
                        if ((edge_server.memory - service.memory_demand) >= 0):
                            edge_server.rf_memory = edge_server.rf_memory + (edge_server.memory - service.memory_demand)
                            #print(edge_server.rf_memory)

def find_shortest_path(origin_network_switch: object, target_network_switch: object) -> int:

    import networkx as nx
    
    """Finds the shortest path (delay used as weight) between two network switches (origin and target).

    Args:
        origin_network_switch (object): Origin network switch.
        target_network_switch (object): Target network switch.

    Returns:
        path (list): Shortest path between the origin and target network switches.
    """
    topology = origin_network_switch.model.topology
    path = []

    if not hasattr(topology, "delay_shortest_paths"):
        topology.delay_shortest_paths = {}

    key = (origin_network_switch, target_network_switch)

    if key in topology.delay_shortest_paths.keys():
        path = topology.delay_shortest_paths[key]
    else:
        path = nx.shortest_path(G=topology, source=origin_network_switch, target=target_network_switch, weight="delay")
        topology.delay_shortest_paths[key] = path

    return path

In [5]:
def stopping_criterion_services(model: object):
    # Defining a variable that will help us to count the number of services successfully provisioned within the infrastructure
    provisioned_services = 0
    
    # Iterating over the list of services to count the number of services provisioned within the infrastructure
    for service in Service.all():

        # Initially, services are not hosted by any server (i.e., their "server" attribute is None).
        # Once that value changes, we know that it has been successfully provisioned inside an edge server.
        if service.server != None:
            provisioned_services += 1
    
    # As EdgeSimPy will halt the simulation whenever this function returns True, its output will be a boolean expression
    # that checks if the number of provisioned services equals to the number of services spawned in our simulation
    stop = False
    if (provisioned_services == Service.count()) | (model.schedule.steps == 60):
        stop = True
    return stop

def stopping_criterion_steps(model: object):    
    # As EdgeSimPy will halt the simulation whenever this function returns True,
    # its output will be a boolean expression that checks if the current time step is 'n'
    n = 60
    return model.schedule.steps == n

In [6]:
# run the experiment many times
print(f">>>>>> [{algo_name}] <<<<<<")

## seed packages
from random import seed
#import torch
import numpy as np

## defining 10 numbers for seed_list
seed_list = [428956419, 1954324947, 1145661099, 1835732737, 794161987, 1329531353, 200496737, 633816299, 1410143363, 1282538739]
#seed_list = [5]
## defining scenarios
#scenario_list = ['2ec_low_on', '2ec_low_off', '2ec_high_on', '2ec_high_off']
#scenario_list = ['sample']
scenario_list = ['2ec_high_off']


# Simulation execution
## 10 different seed to execute 10x the experiment
for seed_value in seed_list:
    print(f"====== [SEED {seed_value}] ======")
    seed(seed_value)
    #torch.manual_seed(seed_value)
    np.random.seed(seed_value)

    ## 4 different scenarios that will be executed by 10 different seeds
    for scenario in scenario_list:
        print(f">>> [scenario {scenario}] <<<")

        if scenario == 'sample':
            mu, sigma = 40, 2.5 # mean and standard deviation
            # start the simulation
            simulator = Simulator(
                tick_duration=1,
                tick_unit="seconds",
                stopping_criterion=stopping_criterion_steps,
                resource_management_algorithm=faticanti,
                logs_directory=f"logs/algorithm={algo_name}/seed={seed_value}/scenario={scenario}",
            )
        else:
            mu, sigma = 15, 5 # mean and standard deviation
            # start the simulation
            simulator = Simulator(
                tick_duration=1,
                tick_unit="seconds",
                stopping_criterion=stopping_criterion_services,
                resource_management_algorithm=faticanti,
                logs_directory=f"logs/algorithm={algo_name}/seed={seed_value}/scenario={scenario}",
            )

        # Loading a sample dataset
        ## change the slas
        path = f"{abspath}/datasets/dataset_{scenario}.json"
        change_slas(path, mu, sigma)
        simulator.initialize(input_file=f"{abspath}/datasets/dataset_{scenario}.json")
        #simulator.initialize(input_file="https://raw.githubusercontent.com/EdgeSimPy/edgesimpy-tutorials/master/datasets/sample_dataset2.json")

        # Assigning the custom collect methods
        User.collect = user_custom_collect_method
        EdgeServer.collect = server_custom_collect_method
        
        # Executing the simulation
        simulator.run_model()

        # Saving Results -------------------------------------------------------------------------------------------------------------------------------------
        logs_edgeserver = pd.DataFrame(simulator.agent_metrics["EdgeServer"])
        logs_edgeserver.to_csv(f"results/{algo_name}_logsserver_{scenario}_{seed_value}.csv", index=False)

        logs_service = pd.DataFrame(simulator.agent_metrics["Service"])
        logs_service.to_csv(f"results/{algo_name}_logsservice_{scenario}_{seed_value}.csv", index=False)
        
        logs_user = pd.DataFrame(simulator.agent_metrics["User"])
        logs_user.to_csv(f"results/{algo_name}_logsuser_{scenario}_{seed_value}.csv", index=False)

>>>>>> [faticanti] <<<<<<
====== [SEED 428956419] ======
>>> [scenario 2ec_high_off] <<<
>>> user 0 <<<
before: {'1': 17.0}
after: {'1': 10.0}
>>> user 1 <<<
before: {'2': 19.0}
after: {'2': 14.0}
>>> user 2 <<<
before: {'3': 26.0}
after: {'3': 16.0}
>>> user 3 <<<
before: {'4': 9.0}
after: {'4': 16.0}
>>> user 4 <<<
before: {'5': 18.0}
after: {'5': 12.0}
>>> user 5 <<<
before: {'6': 7.0}
after: {'6': 11.0}
[STEP 1]
[STEP 2]
[STEP 3]
[STEP 4]
[STEP 5]
[STEP 6]
[STEP 7]
[STEP 8]
[STEP 9]
[STEP 10]
[STEP 11]
[STEP 12]
[STEP 13]
[STEP 14]
[STEP 15]
[STEP 16]
[STEP 17]
[STEP 18]
[STEP 19]
[STEP 20]
[STEP 21]
[STEP 22]
[STEP 23]
[STEP 24]
[STEP 25]
[STEP 26]
[STEP 27]
[STEP 28]
[STEP 29]
[STEP 30]
[STEP 31]
[STEP 32]
[STEP 33]
[STEP 34]
[STEP 35]
[STEP 36]
[STEP 37]
[STEP 38]
[STEP 39]
[STEP 40]
[STEP 41]
[STEP 42]
[STEP 43]
[STEP 44]
[STEP 45]
[STEP 46]
[STEP 47]
[STEP 48]
[STEP 49]
[STEP 50]
[STEP 51]
[STEP 52]
[STEP 53]
[STEP 54]
[STEP 55]
[STEP 56]
[STEP 57]
[STEP 58]
[STEP 59]
[STEP 60